### Install required libraries

In [3]:
%pip install pandas numpy scikit-learn chromadb 

Note: you may need to restart the kernel to use updated packages.


### Import libraries

In [4]:
import pandas as pd
import chromadb

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

### Load dataset

In [5]:
df = pd.read_csv("cybersecurity_intrusion_data.csv")
df.head()

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0


### Prepare dataset

In [6]:
features = df.drop(columns=["session_id","attack_detected"])
features.head()

,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access
0,599,TCP,4,492.983263,DES,0.606818,1,Edge,0
1,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0
2,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0
3,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0
4,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0


### Identify numerical columns

In [7]:
numerical_columns = features.select_dtypes(
    include=["number"]
).columns.tolist()

numerical_columns

['network_packet_size',
 'login_attempts',
 'session_duration',
 'ip_reputation_score',
 'failed_logins',
 'unusual_time_access']

### Identify categorical columns

In [8]:
categorical_columns = features.select_dtypes(
    exclude=["number"]
).columns.tolist()

categorical_columns

['protocol_type', 'encryption_used', 'browser_type']

### Convert the data into numerical vectors

In [9]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

vectors = preprocessor.fit_transform(features)

### Create a ChromaDB collection

In [10]:
client = chromadb.PersistentClient(
    path="./incident_chromadb"
)

collection = client.get_or_create_collection(
    name="incidents",
    metadata={"hnsw:space": "cosine"}
)

print("Collection created!.")

Collection created!.


### Store historical incidents

In [11]:
expected_count = len(vectors)
existing_count = collection.count()
if existing_count == 0:
    batch_size = 5000
    for start in range(0, expected_count, batch_size):
        end = min(start + batch_size, expected_count)
        collection.add(
            ids=[str(i + 1) for i in range(start, end)],
            embeddings=vectors[start:end].tolist(),
            documents=df["session_id"].iloc[start:end].tolist()
        )
        print(f"Inserted {end}/{expected_count}")
    print("Insertion completed.")
else:
    print("Collection already contains data. Skipping insertion.")
print("Total records:", collection.count())

Collection already contains data. Skipping insertion.
Total records: 9537


---

### Test on a new incident

In [16]:
test_incident = pd.DataFrame([{
    "network_packet_size": 512,
    "protocol_type": "TCP",
    "login_attempts": 5,
    "session_duration": 256,
    "encryption_used": "DES",
    "ip_reputation_score": 0.6,
    "failed_logins": 2,
    "browser_type": "Unknown",
    "unusual_time_access": 1
}])

In [17]:
test_incident = test_incident[features.columns]
test_vector = preprocessor.transform(test_incident)

In [18]:
results = collection.query(
    query_embeddings=test_vector.astype(float).tolist(),
    n_results=min(5, collection.count())
)

results

{'ids': [['2927', '1792', '9134', '1065', '9015']],
 'embeddings': None,
 'documents': [['SID_02927',
   'SID_01792',
   'SID_09134',
   'SID_01065',
   'SID_09015']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None, None, None]],
 'distances': [[0.09513002634048462,
   0.0973171591758728,
   0.12688833475112915,
   0.13563776016235352,
   0.13972890377044678]]}

In [19]:
similarity_results = []

for incident_id, document, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    similarity_results.append({
        "Session ID": document.replace("Incident ", ""),
        "Similarity Index (%)": round((1 - distance) * 100, 2)
    })

similarity_df = pd.DataFrame(similarity_results)
similarity_df

,Session ID,Similarity Index (%)
0,SID_02927,90.49
1,SID_01792,90.27
2,SID_09134,87.31
3,SID_01065,86.44
4,SID_09015,86.03
